# Clustering Superconducting Materials into Chemically Distinct Groups
**CSC1171 Data Analytics, Data Mining & Machine Learning · Project notebook for Presentation II**

Group TBC: Malhar Phadke, Federico Singh, Dylan Jackson, Lucy McKee

> **Research question:** Can we use unsupervised machine learning to cluster superconductive materials into chemically or structurally distinct groups?

**Dataset.** The SuperCon database from the National Institute for Materials Science (NIMS), Japan, in the cleaned and featurised version by Chao Zhuang (Kaggle): `featurized.csv`, with 16,521 materials, their critical temperature `Tc`, a known-family label `category`, and 146 descriptors computed from each material's chemical composition.

This notebook carries on from our first presentation:

| Step | What |
|---|---|
| 1–2 | Load, check quality, decide what to clean |
| 3 | Choose clustering features; keep `category` and `Tc` aside |
| 4 | Standardise, then PCA keeping 90% of the variance |
| 5 | Choose K with the elbow method and silhouette score |
| 6–7 | Fit K-Means and describe the clusters (descriptors, elements, Tc) |
| 8 | Compare clusters with the known families |
| 9 | Check reliability: reruns, resampling, 80/85/90% PCA, correlation pruning, a no-structure baseline, excluding records with uncertain oxygen content |
| 10 | HDBSCAN and a Gaussian mixture (soft clustering) |
| 11–13 | Answers to the class questions, limitations, conclusion |

**How to run.** Put this notebook beside `featurized.csv` (or keep the course layout: notebook in `notebooks/`, CSV in `../data/`) and run all cells from top to bottom. It takes a minute or two. Every figure and table is also saved to `project_outputs/` for the slides.

## 0. Setup

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, adjusted_rand_score, normalized_mutual_info_score,
                             homogeneity_score, completeness_score)

RANDOM_STATE = 42   # fixes all randomness, so everyone gets the same results
SIL_SAMPLE = 6000   # silhouette is computed on a fixed random sample of 6,000 rows (all 16,521 is slow)

# Look for the data beside the notebook first, then in a course-style ../data folder.
candidates = [Path("featurized.csv"), Path("../data/featurized.csv"), Path("data/featurized.csv")]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Put featurized.csv beside this notebook (or in ../data/) and run again.")

OUT = Path("project_outputs")
OUT.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})


def save_plot(name):
    """Save the current figure for the slides, then show it."""
    plt.tight_layout()
    plt.savefig(OUT / f"{name}.png", dpi=300, bbox_inches="tight")
    plt.show()


def save_table(table, name):
    """Save a table as CSV for the slides, then show it."""
    table.to_csv(OUT / f"{name}.csv")
    display(table)


print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| data file:", DATA_PATH)

---
## 1. Load the data and take a first look

Our Presentation I notebook (`superconductors_presentation_eda.ipynb`) holds the full exploratory analysis. Here we repeat only the checks that the clustering depends on: size, one row, column types, missing values, duplicates.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(f"{raw.shape[0]:,} materials and {raw.shape[1]} columns")
display(raw.iloc[:5, :8])

# Column types: 3 text columns (formula, composition, category), 1 Boolean, the rest numeric.
raw.dtypes.astype(str).value_counts().rename("number of columns").to_frame()

In [ ]:
ID_COLS = ["formula", "composition"]   # identifiers (text): who the material is
HELD_OUT = ["category", "Tc"]          # kept aside: used only to interpret the clusters afterwards
descriptor_cols = [c for c in raw.columns if c not in ID_COLS + HELD_OUT]
descriptors = raw[descriptor_cols].astype(float)

checks = pd.Series({
    "materials": len(raw),
    "descriptor columns": len(descriptor_cols),
    "missing cells": int(raw.isna().sum().sum()),
    "infinite descriptor values": int(np.isinf(descriptors.to_numpy()).sum()),
    "exact duplicate records": int(raw.duplicated().sum()),
    "repeated formula strings": int(raw["formula"].duplicated().sum()),
    "repeated descriptor vectors (beyond the first)": int(descriptors.duplicated().sum()),
    "materials with Tc = 0 K": int((raw["Tc"] == 0).sum()),
}, name="count").to_frame()
save_table(checks, "data_checks")

---
## 2. Cleaning decisions

We separated **row-wise rules** from steps that learn from the data. The checks above leave very little to clean: no missing or infinite values and no exact duplicate records.

Why a **repeated descriptor vector is not a duplicate**. The descriptors are computed from the chemical composition only, so two records get identical descriptors whenever they have the same composition. The cell below looks at those records directly.

In [ ]:
row_hash = pd.util.hash_pandas_object(descriptors, index=False)
shared = row_hash.duplicated(keep=False).to_numpy()
repeat_groups = raw.loc[shared, ["formula", "composition", "category", "Tc"]].copy()
repeat_groups.insert(0, "group", pd.factorize(row_hash[shared])[0] + 1)

group_summary = repeat_groups.groupby("group").agg(
    records=("formula", "size"), distinct_Tc=("Tc", "nunique"),
    distinct_families=("category", "nunique"), Tc_min=("Tc", "min"), Tc_max=("Tc", "max"))
group_summary["Tc_spread"] = group_summary["Tc_max"] - group_summary["Tc_min"]

print(f"{len(group_summary)} groups share an identical descriptor vector ({len(repeat_groups)} records).")
print(f"Groups whose records have different Tc values:      {(group_summary.distinct_Tc > 1).sum()}")
print(f"Groups whose records have different family labels:  {(group_summary.distinct_families > 1).sum()}")
print(f"Median Tc spread inside a group: {group_summary.Tc_spread.median():.1f} K "
      f"(largest: {group_summary.Tc_spread.max():.1f} K)")

# Three examples with the largest Tc spread
examples = group_summary.nlargest(3, "Tc_spread").index
display(repeat_groups[repeat_groups.group.isin(examples)].sort_values(["group", "Tc"]))

**Why we keep these records.** Identical descriptors mean the same composition, but not necessarily the same experiment: SuperCon collects measurements from many papers. Samples with the same nominal composition can differ in synthesis, oxygen content, pressure or measurement method, all of which change `Tc` without changing the composition-based descriptors. Removing them would throw away real experimental variation. They are not exact duplicates (that would require every column, including `Tc`, to match, and there are none).

One consequence is that records with identical descriptors always land in the same cluster. The clustering cannot tell them apart; only their `Tc` or family label can differ.

---
## 3. Choose the clustering features

For every column we ask the question: *should this information shape the groups?*

- **`formula`, `composition`**: identifiers stored as text. Their chemical content is already encoded in the numerical descriptors. We do use the composition *after* clustering, to list which elements each cluster contains.
- **`category`, `Tc`**: held out. If they entered the clustering, the comparison with known families in Section 8 would be circular (which is a form of target leakage).
- **`compound possible`** is Boolean; it becomes 0/1.
- **Constant columns** carry no information; **identical columns** stored under two names (for example `MagpieData mean NfValence` and `avg f valence electrons`) would count the same quantity twice in every distance, so we keep one copy.

In [ ]:
df = raw.copy()             # all records kept (Section 2)
families = df["category"]   # held out for evaluation only
Tc = df["Tc"]               # held out for evaluation only

X = df[descriptor_cols].astype(float)   # the Boolean becomes 0.0 / 1.0

constant_cols = X.columns[X.nunique() <= 1].tolist()
X = X.drop(columns=constant_cols)

# Find columns that hold the same numbers under two names (allowing for rounding).
corr_before = X.corr()
identical_pairs = []
for i, a in enumerate(corr_before.columns):
    if a in [p[1] for p in identical_pairs]:
        continue
    for b in corr_before.columns[i + 1:]:
        if abs(corr_before.loc[a, b]) > 0.99999 and np.allclose(X[a], X[b], rtol=1e-6, atol=1e-9):
            identical_pairs.append((a, b))
identical = pd.DataFrame(identical_pairs, columns=["kept", "removed (identical values)"]).drop_duplicates("removed (identical values)")
X = X.drop(columns=identical["removed (identical values)"])

print("Constant columns removed:", constant_cols if constant_cols else "none")
save_table(identical, "identical_descriptor_columns")
print(f"Clustering feature matrix: {X.shape[0]:,} materials x {X.shape[1]} descriptors")

---
## 4. Standardise, then reduce the dimension with PCA

K-Means measures **Euclidean distance**. Our descriptors use very different units (atomic weight in the hundreds, fractions between 0 and 1), so without standardisation the large-valued descriptors would dominate every distance. `StandardScaler` gives every descriptor mean 0 and standard deviation 1.

*With many features, distances become less informative; dimensionality reduction (e.g. PCA) is often applied first*. Our descriptors are also strongly redundant (35 pairs with |r| ≥ 0.95 in the EDA). **PCA** rotates the standardised data onto uncorrelated axes ordered by variance; we keep the first components that together explain 90% of the variance.

**What about leakage?** We fitted preprocessing on training data only. That rule protects an evaluation on unseen data. Here no prediction is being evaluated, and `category` and `Tc` never enter the pipeline, so fitting the scaler and PCA on all materials is appropriate. The resampling check in Section 9 refits the whole pipeline on subsets of the data.

In [ ]:
pca_full = Pipeline([("scale", StandardScaler()), ("pca", PCA(svd_solver="full"))])
scores_all = pca_full.fit_transform(X)
explained = pca_full["pca"].explained_variance_ratio_
cumulative = np.cumsum(explained)


def n_components_for(target):
    """Smallest number of principal components whose cumulative variance reaches the target."""
    return int(np.searchsorted(cumulative, target - 1e-12) + 1)


pc_table = pd.DataFrame({"components needed": [n_components_for(t) for t in (0.80, 0.85, 0.90)]},
                        index=pd.Index(["80%", "85%", "90%"], name="variance kept"))
save_table(pc_table, "pca_components_needed")

N_PC = n_components_for(0.90)
Z = scores_all[:, :N_PC]   # the baseline representation used for clustering
print(f"Baseline representation: {Z.shape[0]:,} materials x {N_PC} principal components "
      f"({cumulative[N_PC - 1]:.1%} of the variance). PC1 and PC2 alone: {cumulative[1]:.1%}.")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, len(cumulative) + 1), cumulative * 100, color="#287d8e")
for t, ls in zip((80, 85, 90), (":", "--", "-.")):
    ax.axhline(t, color="grey", ls=ls, lw=1, label=f"{t}% of variance")
ax.axvline(N_PC, color="#d95f02", ls=":", label=f"baseline: {N_PC} PCs")
ax.set(xlabel="Number of principal components", ylabel="Cumulative variance explained (%)",
       ylim=(0, 101), title="How many principal components do we keep?")
ax.legend(loc="lower right")
save_plot("pca_cumulative_variance")

---
## 5. How many clusters? Elbow method and silhouette score

K-Means needs K in advance. We fit K-Means for a range of K (`init="k-means++"`, `n_init=20`, `random_state=42`) and look at two guides:

- **Inertia** (within-cluster sum of squared distances) always falls as K grows; we look for where the improvement levels off (the *elbow*).
- **Mean silhouette score** compares each material's distance to its own cluster with its distance to the nearest other cluster (−1 to 1; higher means better separated).

Both are guides, not answers: interpretability and domain knowledge also count. The known families are **not** used to choose K; we only compare with them afterwards.

In [ ]:
k_values = range(2, 15)
inertia = {1: float(((Z - Z.mean(axis=0)) ** 2).sum())}   # K = 1: all materials in one cluster
silhouette, labels_by_k = {}, {}
for k in k_values:
    km = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=RANDOM_STATE)
    labels_by_k[k] = km.fit_predict(Z)
    inertia[k] = km.inertia_
    silhouette[k] = silhouette_score(Z, labels_by_k[k], sample_size=SIL_SAMPLE, random_state=RANDOM_STATE)

k_table = pd.DataFrame({"inertia": pd.Series(inertia), "mean silhouette": pd.Series(silhouette)}).rename_axis("K")
k_table["inertia drop vs K-1 (%)"] = (-k_table["inertia"].pct_change() * 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_table.index, k_table["inertia"], marker="o")
axes[0].set(xlabel="Number of clusters (K)", ylabel="Inertia", title="Elbow method")
axes[1].plot(list(silhouette), list(silhouette.values()), marker="o", color="#d95f02")
axes[1].set(xlabel="Number of clusters (K)", ylabel="Mean silhouette score", title="Silhouette score by K")
for ax in axes:
    ax.set_xticks(list(k_table.index))
    ax.grid(alpha=0.25)
save_plot("choosing_k")
save_table(k_table.round(3), "choosing_k_table")

**How we choose K.** Two observations guide the choice:

- Silhouette scores are often highest at K = 2 or 3, because the largest split in the data is easy to find. That split would only separate the broadest chemical divisions, which does not answer our question about *distinct groups*.
- Domain knowledge: the dataset names **7 families plus "Other"**. If composition alone separates chemical families, a useful K should be of that order.

**Our rule:** among K = 4 to 10, take the K with the **highest mean silhouette**. It uses only the number of known families as domain knowledge, never the labels themselves. The cell below applies the rule and prints the result; set `K_OVERRIDE` to explore another value. Section 9d checks afterwards whether the conclusions depend on this choice.

**Reading the silhouette value**: above 0.70 strong structure; 0.51–0.70 reasonable; 0.26–0.50 weak structure that could be artificial; 0.25 or below no substantial structure. High-dimensional chemical data with gradual transitions between compositions usually scores in the lower bands, which is why we do not rely on the silhouette alone.

In [ ]:
K_RANGE = range(4, 11)   # domain range: 7 named families + 'Other' (see text above)
K_OVERRIDE = None        # set to a number to explore another K; None applies the rule

K_RULE = max(K_RANGE, key=lambda k: silhouette[k])
K_FINAL = K_OVERRIDE if K_OVERRIDE is not None else K_RULE
best_overall = max(silhouette, key=silhouette.get)
print(f"Highest silhouette for K in {K_RANGE.start}-{K_RANGE.stop - 1}: K = {K_RULE} ({silhouette[K_RULE]:.3f})")
print(f"Highest silhouette over all K tried: K = {best_overall} ({silhouette[best_overall]:.3f})")
print(f"K used from here on: K_FINAL = {K_FINAL}")

In [ ]:
kmeans = KMeans(n_clusters=K_FINAL, init="k-means++", n_init=20, random_state=RANDOM_STATE)
raw_labels = kmeans.fit_predict(Z)

# Cluster IDs are arbitrary (Week 9). Renumber them by size, so cluster 0 is the largest.
size_order = pd.Series(raw_labels).value_counts().index
clusters = pd.Series(raw_labels).map({old: new for new, old in enumerate(size_order)}).to_numpy()

sizes = pd.Series(clusters).value_counts().sort_index()
size_table = pd.DataFrame({"materials": sizes, "% of all": (100 * sizes / sizes.sum()).round(1)}).rename_axis("cluster")
print(f"K-Means with K = {K_FINAL}: inertia {kmeans.inertia_:,.0f}, "
      f"mean silhouette {silhouette_score(Z, clusters, sample_size=SIL_SAMPLE, random_state=RANDOM_STATE):.3f}")
save_table(size_table, "cluster_sizes")

---
## 6. What do the clusters look like?

The left panel colours materials by K-Means cluster, the right panel by known family, both on the first two principal components. PC1 and PC2 show only part of the variance: clusters that overlap here can still be separated along the other components.

In [ ]:
family_order = families.value_counts().index.tolist()
family_colours = dict(zip(family_order, plt.get_cmap("tab10").colors))
cluster_colours = plt.get_cmap("tab10").colors if K_FINAL <= 10 else plt.get_cmap("tab20").colors

rng = np.random.default_rng(RANDOM_STATE)
draw = rng.permutation(len(Z))   # random drawing order, so no group is always hidden underneath

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)
axes[0].scatter(Z[draw, 0], Z[draw, 1], s=5, alpha=0.5, c=[cluster_colours[c] for c in clusters[draw]])
for k in range(K_FINAL):
    cx, cy = Z[clusters == k, 0].mean(), Z[clusters == k, 1].mean()
    axes[0].scatter(cx, cy, s=260, color=cluster_colours[k], edgecolor="black", linewidth=1.2)
    axes[0].annotate(str(k), (cx, cy), ha="center", va="center", fontsize=9, fontweight="bold")
axes[0].legend(handles=[Line2D([], [], marker="o", ls="", color=cluster_colours[k], label=f"cluster {k} (n={sizes[k]:,})")
                        for k in range(K_FINAL)], fontsize=8, loc="best", markerscale=1.2)
axes[0].set_title(f"K-Means clusters (K = {K_FINAL}); large dots are centroids")

axes[1].scatter(Z[draw, 0], Z[draw, 1], s=5, alpha=0.5, c=[family_colours[f] for f in families.to_numpy()[draw]])
axes[1].legend(handles=[Line2D([], [], marker="o", ls="", color=family_colours[f], label=f"{f} (n={(families == f).sum():,})")
                        for f in family_order], fontsize=8, loc="best")
axes[1].set_title("Known families (not used for clustering)")
for ax in axes:
    ax.set(xlabel=f"PC1 ({explained[0]:.1%} of variance)", ylabel=f"PC2 ({explained[1]:.1%} of variance)")
save_plot("pca_clusters_vs_families")

**What we are looking for:**
- Do clusters on the left line up with families on the right? A family that occupies its own region of the plot (for example the cuprates) is a candidate for a clean cluster.
- Which families overlap in this view? Overlap in two dimensions does not prove overlap in all components; Sections 7 and 8 use all of them.
- Does the large "Other" family (the catch-all) occupy one region or several? If several, the clustering may split it into sub-groups that the labels do not name.

---
## 7. Describe the clusters


1. **Descriptor profile**: Analysing which descriptors are unusually high or low in certain clusters. (standard deviation from the mean)
2. **Chemistry**: which elements the materials in each cluster contain. The element list comes from the `composition` text, which the clustering never saw.
3. **Critical temperature**: how `Tc` is distributed in each cluster. `Tc` was never used either, so differences in `Tc` are independent evidence that the clusters capture something physically meaningful.

### 7a. Descriptor profiles

In [ ]:
X_std = pd.DataFrame(pca_full["scale"].transform(X), columns=X.columns)   # z-scores
profile = X_std.groupby(clusters).mean()   # mean z-score of every descriptor in every cluster

# Descriptors whose cluster means differ most, skipping near-copies (|r| > 0.9) of ones already chosen
feature_corr = X.corr()
spread = (profile.max() - profile.min()).sort_values(ascending=False)
top_descriptors = []
for col in spread.index:
    if all(abs(feature_corr.loc[col, chosen]) < 0.9 for chosen in top_descriptors):
        top_descriptors.append(col)
    if len(top_descriptors) == 15:
        break

heat = profile[top_descriptors].T
fig, ax = plt.subplots(figsize=(1.1 * K_FINAL + 4.5, 7))
im = ax.imshow(heat.to_numpy(), cmap="coolwarm", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(K_FINAL), [f"cluster {k}" for k in heat.columns], rotation=0)
ax.set_yticks(range(len(heat)), [c.replace("MagpieData ", "") for c in heat.index])
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.iloc[i, j]
        ax.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=8, color="white" if abs(v) > 1.2 else "black")
fig.colorbar(im, ax=ax, label="cluster mean (standard deviations from overall mean)")
ax.set_title("Descriptors that differ most between clusters")
save_plot("cluster_descriptor_profiles")

In [ ]:
# The same idea in original units for a few descriptors that are easy to explain (medians, robust to skew)
readable = {
    "0-norm": "number of elements",
    "MagpieData mean AtomicWeight": "mean atomic weight",
    "MagpieData mean Electronegativity": "mean electronegativity",
    "MagpieData range Electronegativity": "electronegativity range",
    "max ionic char": "max ionic character",
    "transition metal fraction": "transition-metal fraction",
    "avg d valence electrons": "avg d valence electrons",
    "avg f valence electrons": "avg f valence electrons",
}
readable = {c: name for c, name in readable.items() if c in df.columns}
median_profile = df[list(readable)].groupby(clusters).median().rename(columns=readable).rename_axis("cluster")
save_table(median_profile.round(2), "cluster_median_descriptors")

### 7b. Which elements does each cluster contain?

In [ ]:
element_sets = df["composition"].map(lambda s: set(re.findall(r"[A-Z][a-z]?", s)))
element_counts = pd.Series([e for s in element_sets for e in s]).value_counts()
common_elements = element_counts[element_counts >= 50].index   # ignore very rare elements

presence = pd.DataFrame({e: element_sets.map(lambda s, e=e: e in s) for e in common_elements})
pct_by_cluster = presence.groupby(clusters).mean() * 100   # % of materials in each cluster containing each element
overall_pct = presence.mean() * 100

# Heatmap for the 18 most common elements overall
show = element_counts.index[:18]
heat = pct_by_cluster[show].T
fig, ax = plt.subplots(figsize=(1.0 * K_FINAL + 3.5, 6.5))
im = ax.imshow(heat.to_numpy(), cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(K_FINAL), [f"cluster {k}" for k in heat.columns])
ax.set_yticks(range(len(show)), show)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.iloc[i, j]:.0f}", ha="center", va="center", fontsize=8,
                color="white" if heat.iloc[i, j] > 60 else "black")
fig.colorbar(im, ax=ax, label="% of the cluster's materials containing the element")
ax.set_title("Element presence by cluster (18 most common elements)")
save_plot("cluster_element_presence")

# Signature elements: present in at least 25% of a cluster and most over-represented relative to the whole dataset
signature_rows = []
for k in range(K_FINAL):
    share = pct_by_cluster.loc[k]
    lift = (share / overall_pct)[share >= 25].sort_values(ascending=False)
    signature_rows.append({
        "cluster": k,
        "signature elements (% of cluster)": ", ".join(f"{e} ({share[e]:.0f}%)" for e in lift.index[:5]),
        "example formulas": ", ".join(df.loc[clusters == k, "formula"].sample(3, random_state=RANDOM_STATE)),
    })
signatures = pd.DataFrame(signature_rows).set_index("cluster")
save_table(signatures, "cluster_signature_elements")

### 7c. Critical temperature by cluster (Tc was not used for clustering)

In [ ]:
tc_by_cluster = Tc.groupby(clusters).describe()[["count", "mean", "25%", "50%", "75%", "max"]]
tc_by_cluster["% above 77 K"] = (Tc > 77).groupby(clusters).mean() * 100   # 77 K: liquid nitrogen
tc_by_cluster = tc_by_cluster.rename(columns={"50%": "median"}).rename_axis("cluster")
save_table(tc_by_cluster.round(1), "cluster_tc_summary")

fig, ax = plt.subplots(figsize=(9, 0.55 * K_FINAL + 1.8))
horizontal = {"orientation": "horizontal"} if tuple(int(v) for v in matplotlib.__version__.split(".")[:2]) >= (3, 10) else {"vert": False}
ax.boxplot([Tc[clusters == k] for k in range(K_FINAL)], widths=0.6, **horizontal,
           tick_labels=[f"cluster {k} (n={sizes[k]:,})" for k in range(K_FINAL)],
           flierprops={"markersize": 2, "alpha": 0.3})
ax.axvline(77, color="#d95f02", ls="--", lw=1, label="77 K (liquid nitrogen)")
ax.invert_yaxis()
ax.set(xlabel="Critical temperature Tc (K)", title="Tc distribution in each cluster")
ax.legend(loc="lower right")
save_plot("cluster_tc_boxplot")

**How to read Section 7.**
- In the descriptor heatmap, red cells mean the cluster's average is above the dataset average (in standard deviations), blue below. A cluster is chemically distinct if it differs on several descriptors at once.
- In the element table, a column with values near 100 for a few elements identifies the cluster's chemistry: Cu and O together point to cuprate-like materials; Fe with As or Se to iron-based materials; B with C (and Ni) to borocarbides. The *signature elements* and *example formulas* let us give each cluster a descriptive name, which is more informative than its number.
- In the Tc table, compare medians and the share above 77 K. Because `Tc` was not used for clustering, clear differences between clusters are evidence that the grouping captures something physically relevant. Similar Tc distributions in two clusters do not make them chemically similar.

---
## 8. Do the clusters match the known superconductor families?

We bring back the `category` labels. A **contingency table** counts how many materials of each family fall into each cluster. Three summary scores help, chosen because they aid in comparing the two groupings:

- **Adjusted Rand Index (ARI)**: 1 means the two groupings are identical; about 0 means no better than random assignment.
- **Normalised mutual information (NMI)**: 0 to 1; how much knowing the cluster tells us about the family.
- **Homogeneity / completeness**: does each cluster contain only one family? Does each family fall into only one cluster?

The `Other` family (41% of materials) is a catch-all of everything that is not in the named families, so we also report the scores without it. A shuffled-label baseline shows what pure chance gives.

In [ ]:
ct = pd.crosstab(families, clusters).reindex(family_order)
ct.columns.name, ct.index.name = "cluster", "family"
save_table(ct, "family_by_cluster_counts")

family_share = ct.div(ct.sum(axis=1), axis=0) * 100   # % of each family in each cluster
fig, ax = plt.subplots(figsize=(1.0 * K_FINAL + 5, 4.8))
im = ax.imshow(family_share.to_numpy(), cmap="Greens", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(K_FINAL), [f"cluster {k}" for k in family_share.columns])
ax.set_yticks(range(len(family_order)), [f"{f} (n={ct.loc[f].sum():,})" for f in family_order])
for i in range(family_share.shape[0]):
    for j in range(family_share.shape[1]):
        v = family_share.iloc[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > 60 else "black")
fig.colorbar(im, ax=ax, label="% of the family's materials")
ax.set_title("Where does each known family end up? (rows sum to 100%)")
save_plot("family_by_cluster_heatmap")

In [ ]:
def agreement(cluster_labels, family_labels):
    """Scores comparing a clustering with the known families."""
    return {"ARI": adjusted_rand_score(family_labels, cluster_labels),
            "NMI": normalized_mutual_info_score(family_labels, cluster_labels),
            "homogeneity": homogeneity_score(family_labels, cluster_labels),
            "completeness": completeness_score(family_labels, cluster_labels)}


named = (families != "Other").to_numpy()
shuffled = np.random.default_rng(RANDOM_STATE).permutation(clusters)
agreement_table = pd.DataFrame({
    "all materials": agreement(clusters, families),
    "named families only (without 'Other')": agreement(clusters[named], families[named]),
    "baseline: shuffled cluster labels": agreement(shuffled, families),
}).T
save_table(agreement_table.round(3), "agreement_with_families")

family_summary = pd.DataFrame({
    "materials": ct.sum(axis=1),
    "main cluster": ct.idxmax(axis=1),
    "% in main cluster": family_share.max(axis=1).round(1),
})
cluster_makeup = (ct / ct.sum(axis=0) * 100).round(1)   # family mix inside each cluster (columns sum to 100%)
cluster_makeup.index.name = "% of cluster that is..."
save_table(family_summary, "family_main_cluster")
save_table(cluster_makeup, "cluster_family_makeup")

**Results from Section 8.**
- In the heatmap, each row is a family and sums to 100%. A family concentrated in one column is *recovered* by the clustering. A family spread over several columns is split by composition, which can be a real finding (sub-families) or a sign that the descriptors do not capture what defines the family.
- "Other" is a catch-all, so we expect it to spread across several clusters. A cluster made mostly of "Other" materials is interesting: its signature elements (Section 7b) may describe a group of materials the labels do not name.
- **ARI / NMI:** compare with the shuffled baseline (about 0). Values clearly above 0 mean the clusters carry real information about the families; values near 1 would mean the clustering reproduces them exactly. High homogeneity with lower completeness means the clusters are *purer than the families*: families are split into several clean clusters rather than mixed together.

---
## 9. Are the clusters reliable, or just "clumps"?

A class question asked how we know the clusters are not *clumps of useless information*. The Week 9 limitations tell us what to test: K-Means **always** returns K clusters, results can depend on **initialisation**, on the **sample**, and on the **features and scaling**. We check each in turn, comparing every new clustering with our baseline using the ARI (1 = identical grouping).

### 9a. Different starting centroids

In [ ]:
seed_rows = []
for seed in range(10):
    for n_init in (1, 20):
        labels = KMeans(n_clusters=K_FINAL, init="k-means++", n_init=n_init, random_state=seed).fit_predict(Z)
        seed_rows.append({"n_init": n_init, "seed": seed, "ARI vs baseline": adjusted_rand_score(clusters, labels)})
seed_results = pd.DataFrame(seed_rows)
seed_summary = seed_results.groupby("n_init")["ARI vs baseline"].agg(["mean", "min", "max"])
seed_summary.index = [f"n_init = {n} (10 different seeds)" for n in seed_summary.index]
save_table(seed_summary.round(3), "stability_initialisation")

### 9b. Resampling: would a different sample of materials give the same groups?

We draw 20 random subsets of 80% of the materials, refit the **whole pipeline** (scaler, PCA, K-Means) on each subset, assign all materials with that refitted pipeline, and compare with the baseline.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
resample_ari = []
for r in range(20):
    rows = rng.choice(len(X), size=int(0.8 * len(X)), replace=False)
    pipe = Pipeline([("scale", StandardScaler()),
                     ("pca", PCA(n_components=N_PC, svd_solver="full")),
                     ("kmeans", KMeans(n_clusters=K_FINAL, init="k-means++", n_init=10, random_state=RANDOM_STATE))])
    pipe.fit(X.iloc[rows])
    resample_ari.append(adjusted_rand_score(clusters, pipe.predict(X)))
resample_ari = np.array(resample_ari)
print(f"ARI vs baseline over 20 resamples: mean {resample_ari.mean():.3f}, "
      f"min {resample_ari.min():.3f}, max {resample_ari.max():.3f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(resample_ari, bins=np.linspace(0, 1, 21), color="#287d8e", edgecolor="white")
ax.set(xlabel="ARI with the baseline clustering", ylabel="Resamples", title="Stability under resampling (20 x 80% subsets)")
save_plot("stability_resampling")

### 9c. Does the representation matter? (80/85/90% PCA, no PCA, correlation pruning)

This was a planned comparison from our *Next steps* slide. For correlation pruning we drop one descriptor from every pair with |r| ≥ 0.95 before scaling and PCA. Silhouette scores are computed in each variant's own space, so they are only roughly comparable across rows; the ARI columns compare the groupings directly.

In [ ]:
def correlation_prune(features, threshold=0.95):
    """Drop the later column of every pair whose absolute correlation reaches the threshold."""
    c = features.corr().abs()
    upper = c.where(np.triu(np.ones(c.shape, dtype=bool), k=1))
    drop = [col for col in upper.columns if (upper[col] >= threshold).any()]
    return features.drop(columns=drop), drop


X_pruned, pruned_cols = correlation_prune(X)
print(f"Correlation pruning keeps {X_pruned.shape[1]} of {X.shape[1]} descriptors.")

variants = {
    "PCA 80%": (X, 0.80),
    "PCA 85%": (X, 0.85),
    "PCA 90% (baseline)": (X, 0.90),
    "all descriptors, no PCA": (X, None),
    "correlation-pruned + PCA 90%": (X_pruned, 0.90),
}
variant_rows, variant_labels = [], {}
for name, (features, variance) in variants.items():
    steps = [("scale", StandardScaler())]
    if variance is not None:
        steps.append(("pca", PCA(n_components=variance, svd_solver="full")))
    space = Pipeline(steps).fit_transform(features)
    labels = KMeans(n_clusters=K_FINAL, init="k-means++", n_init=20, random_state=RANDOM_STATE).fit_predict(space)
    variant_labels[name] = labels
    variant_rows.append({
        "variant": name, "dimensions": space.shape[1],
        "silhouette (own space)": silhouette_score(space, labels, sample_size=SIL_SAMPLE, random_state=RANDOM_STATE),
        "ARI vs baseline": adjusted_rand_score(clusters, labels),
        "ARI vs families": adjusted_rand_score(families, labels),
        "NMI vs families": normalized_mutual_info_score(families, labels),
    })
variant_table = pd.DataFrame(variant_rows).set_index("variant")
save_table(variant_table.round(3), "sensitivity_representation")

### 9d. Is the agreement with the families an artefact of our choice of K?

The families were not used to pick K. Looking back, we can still check how agreement with the families changes with K, using the clusterings already fitted in Section 5.

In [ ]:
k_agreement = pd.DataFrame({k: {"ARI vs families": adjusted_rand_score(families, lab),
                                "NMI vs families": normalized_mutual_info_score(families, lab),
                                "mean silhouette": silhouette[k]}
                            for k, lab in labels_by_k.items()}).T.rename_axis("K")

fig, ax = plt.subplots(figsize=(7.5, 4))
for col, marker in zip(k_agreement.columns, ("o", "s", "^")):
    ax.plot(k_agreement.index, k_agreement[col], marker=marker, label=col)
ax.axvline(K_FINAL, color="grey", ls=":", label=f"chosen K = {K_FINAL}")
ax.set(xlabel="Number of clusters (K)", ylabel="Score", title="Agreement with known families across K")
ax.set_xticks(list(k_agreement.index))
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
save_plot("agreement_across_k")
save_table(k_agreement.round(3), "agreement_across_k")

### 9e. Would K-Means find "clusters" in data with no group structure?

K-Means *always* returns K clusters. To see what a meaningless clustering scores, we build a **no-structure** version of our data: each principal component is shuffled independently. Every axis keeps exactly the same spread of values, but any groups formed by *combinations* of axes are destroyed. If our real silhouette is clearly higher than on this shuffled data, the clusters reflect structure that does not arise by chance.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
null_silhouettes = []
for r in range(5):
    Z_null = np.column_stack([rng.permutation(Z[:, j]) for j in range(Z.shape[1])])
    null_labels = KMeans(n_clusters=K_FINAL, init="k-means++", n_init=10, random_state=RANDOM_STATE).fit_predict(Z_null)
    null_silhouettes.append(silhouette_score(Z_null, null_labels, sample_size=SIL_SAMPLE, random_state=RANDOM_STATE))

real_silhouette = silhouette_score(Z, clusters, sample_size=SIL_SAMPLE, random_state=RANDOM_STATE)
null_table = pd.DataFrame({"mean silhouette": [real_silhouette, np.mean(null_silhouettes)],
                           "spread (SD over 5 shuffles)": [np.nan, np.std(null_silhouettes)]},
                          index=["real data", "no-structure data (components shuffled)"])
save_table(null_table.round(3), "no_structure_baseline")

### 9f. Do records with uncertain oxygen content shape the clusters?

Many formulas in SuperCon, mostly cuprates, give the oxygen content as a variable instead of a number (for example `O7-Z` or `O8+Y`), because the exact amount depends on how each sample was made and was not measured. The descriptors for these records are computed from a nominal composition that may not match the real sample. We flag every record whose formula writes oxygen this way, refit the **whole pipeline** (scaler, PCA, K-Means) on the remaining records only, and compare with the baseline.

Cluster numbers are arbitrary, so each new cluster takes the number of the baseline cluster it shares the most materials with. This way the numbers and colours below match those in Sections 6 and 7b.

In [ ]:
from scipy.optimize import linear_sum_assignment

# Flag formulas that write oxygen as a variable (e.g. O7-Z, O8+Y, O6.5-d, Oz) instead of a number.
# 'Os' (osmium) and 'Og' are elements, so a lowercase s or g straight after O is not a variable.
OXYGEN_VARIABLE = r"O(?![sg])[\d.]*(?:[+-]|[a-z])"
oxygen_flag = df["formula"].str.contains(OXYGEN_VARIABLE, regex=True).to_numpy()
known = ~oxygen_flag   # records with a stated oxygen content, or no oxygen at all

print(f"{oxygen_flag.sum():,} records flagged ({oxygen_flag.mean():.1%}); {known.sum():,} records kept.")
if oxygen_flag.any():
    display(df.loc[oxygen_flag, ["formula", "category", "Tc"]]
              .sample(min(5, int(oxygen_flag.sum())), random_state=RANDOM_STATE))
    display(families[oxygen_flag].value_counts().rename("flagged records").to_frame())
else:
    print("No formulas match the rule: check how oxygen is written in the `formula` column.")

# Same pipeline as the baseline, fitted on the kept records only
pipe_known = Pipeline([("scale", StandardScaler()),
                       ("pca", PCA(n_components=N_PC, svd_solver="full")),
                       ("kmeans", KMeans(n_clusters=K_FINAL, init="k-means++", n_init=20, random_state=RANDOM_STATE))])
raw_known = pipe_known.fit_predict(X[known])

# Give each new cluster the number of the baseline cluster it overlaps most
overlap = pd.crosstab(raw_known, clusters[known]).reindex(index=range(K_FINAL), columns=range(K_FINAL), fill_value=0)
new_ids, old_ids = linear_sum_assignment(-overlap.to_numpy())
labels_known = pd.Series(raw_known).map(dict(zip(new_ids, old_ids))).to_numpy()

oxygen_table = pd.Series({
    "records kept": f"{known.sum():,} of {len(X):,}",
    "ARI vs baseline (kept records)": f"{adjusted_rand_score(clusters[known], labels_known):.3f}",
    "kept records that change cluster": f"{(labels_known != clusters[known]).mean():.1%}",
    "ARI vs families, baseline (kept records)": f"{adjusted_rand_score(families[known], clusters[known]):.3f}",
    "ARI vs families, oxygen records excluded": f"{adjusted_rand_score(families[known], labels_known):.3f}",
}, name="value").to_frame()
save_table(oxygen_table, "stability_oxygen_excluded")

# Element presence and signature elements for the new clusters (same method as Section 7b)
presence_known = presence[known].reset_index(drop=True)
pct_known = presence_known.groupby(labels_known).mean() * 100
overall_known = presence_known.mean() * 100
formulas_known = df.loc[known, "formula"].reset_index(drop=True)

rows = []
for k in range(K_FINAL):
    share = pct_known.loc[k]
    lift = (share / overall_known)[share >= 25].sort_values(ascending=False)
    rows.append({
        "cluster": k,
        "signature elements (% of cluster)": ", ".join(f"{e} ({share[e]:.0f}%)" for e in lift.index[:5]),
        "example formulas": ", ".join(formulas_known[labels_known == k].sample(3, random_state=RANDOM_STATE)),
    })

In [ ]:
# PCA plot after excluding flagged oxygen records
Z_known = pipe_known[:-1].transform(X[known])
families_known = families[known].to_numpy()
explained_known = pipe_known["pca"].explained_variance_ratio_
sizes_known = pd.Series(labels_known).value_counts().sort_index()
draw = np.random.default_rng(RANDOM_STATE).permutation(len(Z_known))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)

axes[0].scatter(Z_known[draw, 0], Z_known[draw, 1], s=5, alpha=0.5,
                c=[cluster_colours[k] for k in labels_known[draw]])

for k in range(K_FINAL):
    cx, cy = Z_known[labels_known == k, :2].mean(axis=0)
    axes[0].scatter(cx, cy, s=260, color=cluster_colours[k],
                    edgecolor="black", linewidth=1.2)
    axes[0].annotate(str(k), (cx, cy), ha="center", va="center", fontweight="bold")

axes[0].legend(handles=[
    Line2D([], [], marker="o", ls="", color=cluster_colours[k],
           label=f"cluster {k} (n={sizes_known[k]:,})")
    for k in range(K_FINAL)], fontsize=8)
axes[0].set_title("K-Means clusters — oxygen records excluded")

fam_draw = draw[np.argsort(families_known[draw] != "Other", kind="stable")]
axes[1].scatter(Z_known[fam_draw, 0], Z_known[fam_draw, 1], s=5, alpha=0.5,
                c=[family_colours[f] for f in families_known[fam_draw]])
axes[1].legend(handles=[
    Line2D([], [], marker="o", ls="", color=family_colours[f],
           label=f"{f} (n={(families_known == f).sum():,})")
    for f in family_order], fontsize=8)
axes[1].set_title("Known families — oxygen records excluded")

for ax in axes:
    ax.set(xlabel=f"PC1 ({explained_known[0]:.1%} variance)",
           ylabel=f"PC2 ({explained_known[1]:.1%} variance)")

save_plot("pca_clusters_vs_families_oxygen_known")

In [ ]:
# Element presence heatmap — same format as Section 7b
heat = pct_known[element_counts.index[:18]].T

fig, ax = plt.subplots(figsize=(1.0 * K_FINAL + 3.5, 6.5))
im = ax.imshow(heat, cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(K_FINAL), [f"cluster {k}" for k in heat.columns])
ax.set_yticks(range(len(heat)), heat.index)

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        value = heat.iloc[i, j]
        ax.text(j, i, f"{value:.0f}", ha="center", va="center",
                fontsize=8, color="white" if value > 60 else "black")

fig.colorbar(im, ax=ax, label="% of cluster containing element")
ax.set_title("Element presence by cluster — oxygen records excluded")
save_plot("cluster_elements_oxygen_known")

In [ ]:
# Signature-elements table — already calculated in Section 9f
save_table(pd.DataFrame(rows).set_index("cluster"),
           "cluster_signature_elements_oxygen_known")

**How to read Section 9** (ARI ranges are rough guides, not formal thresholds):
- **9a and 9b:** a mean ARI above about 0.9 means the same groups come back almost exactly; 0.7–0.9 means mostly the same groups with some boundary materials moving; well below 0.7 means the grouping depends on chance. With `n_init=1`, a single unlucky start can give a worse solution; `n_init=20` keeps the best of 20 starts, so its ARIs should be higher and less variable.
- **9c:** if the ARI against the baseline stays high across the 80/85/90% PCA, no-PCA and pruned variants, our conclusions do not depend on these preprocessing choices.
- **9d:** if agreement with the families is similar over a range of K, our conclusions do not hinge on the exact K.
- **9e:** if the real silhouette clearly exceeds the no-structure silhouette, the clusters are more than K-Means' habit of always returning K groups.
- **9f:** if the ARI against the baseline stays high and each cluster keeps the same signature elements, the clusters do not depend on records with uncertain oxygen content. Compare the plots and tables with Sections 6 and 7b: a cluster that shrinks, merges with another or changes its signature elements was partly built on those records. The flagged-records table shows which families lose the most materials; we expect mostly cuprates.

---
## 10. Extension: two other kinds of clustering


### 10a. HDBSCAN: clusters as dense regions, with a "noise" label

K-Means assumes roughly round, similarly sized clusters and must place every material in some cluster. HDBSCAN instead looks for **dense regions** of any shape, does not need K, and labels materials in sparse regions as **noise** (`-1`). Density estimates become unreliable in many dimensions, so we run it on the first 10 principal components. `min_cluster_size=100` means a group must contain at least 100 materials (about 0.6% of the data).

In [ ]:
try:
    from sklearn.cluster import HDBSCAN   # needs scikit-learn 1.3 or newer
except ImportError:
    HDBSCAN = None
    print("HDBSCAN needs scikit-learn >= 1.3: run `pip install -U scikit-learn`. Skipping this section.")

if HDBSCAN is not None:
    hdb = HDBSCAN(min_cluster_size=100, min_samples=10, copy=True)
    hdb_labels = hdb.fit_predict(Z[:, :10])
    is_noise = hdb_labels == -1
    n_hdb = len(set(hdb_labels)) - (1 if is_noise.any() else 0)
    print(f"HDBSCAN found {n_hdb} dense clusters; {is_noise.mean():.1%} of materials are labelled noise.")

    # One row per HDBSCAN cluster: its size, the family it mostly contains, and the matching K-Means cluster
    found = pd.DataFrame({"hdbscan": hdb_labels, "family": families.to_numpy(), "kmeans": clusters})[~is_noise]
    hdb_summary = found.groupby("hdbscan").agg(
        materials=("family", "size"),
        main_family=("family", lambda s: s.value_counts().index[0]),
        pct_main_family=("family", lambda s: round(100 * s.value_counts(normalize=True).iloc[0], 1)),
        main_kmeans_cluster=("kmeans", lambda s: s.value_counts().index[0]),
    ).sort_values("materials", ascending=False)
    save_table(hdb_summary, "hdbscan_clusters")

    hdb_scores = pd.DataFrame({
        "HDBSCAN (non-noise materials)": agreement(hdb_labels[~is_noise], families[~is_noise]),
        "K-Means (same materials)": agreement(clusters[~is_noise], families[~is_noise]),
    }).T
    save_table(hdb_scores.round(3), "hdbscan_vs_kmeans")

    noise_by_family = (pd.Series(is_noise).groupby(families.to_numpy()).mean() * 100).reindex(family_order)
    save_table(noise_by_family.round(1).rename("% labelled noise").to_frame(), "hdbscan_noise_by_family")

### 10b. Gaussian mixture: soft clustering

K-Means gives each material exactly one cluster (*hard* clustering). A **Gaussian mixture model** describes each cluster as a bell-shaped (Gaussian) cloud with its own shape and orientation, and gives every material a **probability of belonging to each cluster** (*soft* clustering). Materials with no dominant probability sit between groups. We use the same K and the same 90% PCA space as K-Means.

In [ ]:
gmm = GaussianMixture(n_components=K_FINAL, covariance_type="full", n_init=3, random_state=RANDOM_STATE)
gmm_labels = gmm.fit_predict(Z)
membership = gmm.predict_proba(Z)     # one probability per cluster; each row sums to 1
max_prob = membership.max(axis=1)     # probability of the most likely cluster

soft_table = pd.DataFrame({
    "value": [adjusted_rand_score(clusters, gmm_labels),
              adjusted_rand_score(families, gmm_labels),
              normalized_mutual_info_score(families, gmm_labels),
              (max_prob >= 0.9).mean() * 100,
              (max_prob < 0.6).mean() * 100]},
    index=["ARI: Gaussian mixture vs K-Means", "ARI: Gaussian mixture vs families", "NMI: Gaussian mixture vs families",
           "% of materials with a clear cluster (max probability >= 0.9)",
           "% of materials split between clusters (max probability < 0.6)"])
save_table(soft_table.round(3), "soft_clustering_summary")

by_family = pd.Series(max_prob).groupby(families.to_numpy()).agg(["mean", lambda s: (s < 0.6).mean() * 100])
by_family.columns = ["mean max probability", "% split between clusters"]
save_table(by_family.reindex(family_order).round(3), "soft_clustering_by_family")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(max_prob, bins=40, color="#287d8e", edgecolor="white")
ax.set(xlabel="Probability of the most likely cluster", ylabel="Materials",
       title="How certain is each material's cluster? (Gaussian mixture)")
save_plot("soft_clustering_certainty")

**How to read Section 10.**
- **HDBSCAN** answers a different question from K-Means: *where are the dense groups?* Small, very pure HDBSCAN clusters are tight chemical sub-families (often one compound family studied with many dopings). A large noise share means much of the data lies in sparse regions rather than in compact clumps; for that data a hard partition such as K-Means imposes boundaries that are partly arbitrary.
- **Gaussian mixture:** if most materials have a maximum membership probability near 1, the groups are well separated in this space; materials with low maximum probability sit between groups. The by-family table shows which families sit on boundaries. Note that a Gaussian mixture in 20+ dimensions often gives very confident probabilities even when clusters touch, so these numbers are best compared *between* families rather than read as absolute certainty.

---
## 11. Answers to the questions from Presentation I

**1. Why did you choose unsupervised learning?**
Our question is whether materials *form* distinct groups, not how well we can predict a given label. A classifier trained on `category` could only reproduce the existing families, including the vague "Other" (41% of the data). Clustering lets the composition descriptors speak for themselves: we can then test whether the groups they form agree with the known families (Section 8) and whether they reveal sub-groups that the labels do not name (Section 7b). The labels are still used, but only afterwards, as a check.

**2. How do you get reliable results, and not just clumps of useless information?**
Four kinds of evidence, all in this notebook: (i) **internal**: elbow and silhouette (Section 5), compared with a no-structure baseline (9e); (ii) **stability**: the same groups return under different starting centroids (9a), different samples of materials (9b), different preprocessing choices (9c), nearby values of K (9d) and with records of uncertain oxygen content left out (9f); (iii) **external**: agreement with the known families, against a shuffled baseline (Section 8); (iv) **independent physical meaning**: `Tc` was never used, yet we compare its distribution between clusters (7c), and every cluster is described by its chemistry (7a–b), so a "useless clump" would show up as a cluster with no recognisable profile.

**3. Would soft clustering reveal interesting results?**
We tried it with a Gaussian mixture model (Section 10b). Its membership probabilities show which materials, and which families, sit between groups.

**4. Why is a repeated descriptor vector not necessarily a duplicate?**
The descriptors are computed from composition only. Two records with the same composition get identical descriptors even when they are separate measurements with different `Tc` (Section 2 shows examples). An exact duplicate would match in every column, including `Tc`; the data contain none.

---
## 12. Limitations and good practice

- **Composition only, not structure.** Every descriptor is computed from the chemical formula. Even the `SpaceGroupNumber` statistics describe the *pure elements*, not each compound's crystal structure. So we can answer the *chemically distinct* half of our research question; *structurally distinct* groups would need crystal-structure data, which this dataset does not contain.
- **Dataset bias.** SuperCon reflects which materials researchers chose to study. Cuprates and iron-based materials are heavily represented (often one parent compound with many dopings), which favours clusters around them, while rare families (organic: 75, elemental: 64, hydrogen-rich: 6 materials) have little influence on K-Means and are unlikely to get clusters of their own.
- **K-Means assumptions** (Week 9): it prefers round, similarly sized clusters and always returns K groups. Sections 9e and 10 test these assumptions with a no-structure baseline, a density-based method and a mixture model.
- **Distances in many dimensions** become less informative; PCA reduces this, but the result still depends on which descriptors exist. Section 9c shows how much the result depends on these choices.
- **The family labels are not ground truth.** They are coarse, human-defined categories, and "Other" is a mixture. Disagreement with them is not automatically a failure of the clustering.
- **A cluster is a mathematical result, not automatically a scientific discovery** (Week 9). Any new grouping suggested here would need to be checked against the materials-science literature.

---
## 13. Conclusion

The cell below collects the key numbers from this notebook, so the summary always matches the current run.

In [ ]:
summary = pd.Series({
    "materials clustered": f"{len(Z):,}",
    "descriptors after removing identical columns": X.shape[1],
    "principal components kept (90% of variance)": N_PC,
    "number of clusters K": K_FINAL,
    "mean silhouette (real data)": f"{real_silhouette:.3f}",
    "mean silhouette (no-structure baseline)": f"{np.mean(null_silhouettes):.3f}",
    "ARI vs families, all materials": f"{agreement_table.loc['all materials', 'ARI']:.3f}",
    "ARI vs families, named families only": f"{agreement_table.iloc[1]['ARI']:.3f}",
    "NMI vs families, all materials": f"{agreement_table.loc['all materials', 'NMI']:.3f}",
    "stability: mean ARI over 20 resamples": f"{resample_ari.mean():.3f}",
    "stability: mean ARI over 10 seeds (n_init=20)": f"{seed_summary.iloc[-1]['mean']:.3f}",
}, name="value").to_frame()
save_table(summary, "key_numbers")

**Answer to the research question.** *[Group: write 3–5 sentences from the key numbers above and the descriptions in Sections 7–9: which chemically distinct groups did K-Means find, which known families did they recover, which were split or merged, how stable were they, and what does the Tc comparison add?]*

---
### Sources and acknowledgements

- **Data:** SuperCon database, National Institute for Materials Science (NIMS), Japan. Cleaned and featurised version: *SUPERCON Dataset: A Superconductor Materials Database from NIMS*, Chao Zhuang, Kaggle.
- **Descriptors:** Magpie elemental-property statistics, Ward, L., Agrawal, A., Choudhary, A. & Wolverton, C. (2016). A general-purpose machine learning framework for predicting properties of inorganic materials. *npj Computational Materials* 2, 16028; computed with matminer (Ward et al., 2018, *Computational Materials Science* 152, 60–69).
- **Methods:** Lloyd (1982), *IEEE Trans. Inf. Theory* 28(2); Arthur & Vassilvitskii (2007), k-means++, *Proc. SODA*; Rousseeuw (1987), silhouettes, *J. Comput. Appl. Math.* 20; Kaufman & Rousseeuw (1990), *Finding Groups in Data*, Wiley; Hubert & Arabie (1985), Adjusted Rand Index, *J. Classification* 2; Campello, Moulavi & Sander (2013), HDBSCAN, *PAKDD*; Jain (2010), *Pattern Recognition Letters* 31(8).
- **Course material:** CSC1171 lectures and notebooks, Weeks 3–9 (N. Chinichian, 2026).

**AI use disclosure** (as promised on our ethics slide): parts of this notebook's code and text were drafted with an AI assistant (Claude, Anthropic). *[Group: before submitting, add how you reviewed, ran and checked the code and the interpretations.]*